# 04. Model Benchmarking Suite
Training and cross-validation of 7 ML models for AML transaction monitoring.

In [1]:
#loading the data from the previous data 
import scipy.sparse
import numpy as np

# 1. Load balanced training data (produced by BorderlineSMOTE)
X_train = scipy.sparse.load_npz("../data/processed/X_train_resampled.npz")
y_train = np.load("../data/processed/y_train_resampled.npy")

# 2. Load original test data (evaluated on raw real-world distribution)
X_test  = scipy.sparse.load_npz("../data/processed/X_test_selected.npz")
y_test  = np.load("../data/processed/y_test.npy")

# Verification
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Test shapes:  X={X_test.shape},  y={y_test.shape}")

Train shapes: X=(15191966, 86505), y=(15191966,)
Test shapes:  X=(1900971, 86505),  y=(1900971,)


In [1]:
import scipy.sparse
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import RandomizedSearchCV

# Load data
X_train = scipy.sparse.load_npz("../data/processed/X_train_resampled.npz")
y_train = np.load("../data/processed/y_train_resampled.npy")
X_test  = scipy.sparse.load_npz("../data/processed/X_test_selected.npz")
y_test  = np.load("../data/processed/y_test.npy")

benchmark_results = []

def evaluate_and_record(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)

    metrics = {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
        "F1-Score": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0)
    }
    benchmark_results.append(metrics)
    print(pd.DataFrame([metrics]).to_string(index=False))

In [2]:
from sklearn.ensemble import RandomForestClassifier

param_dist = {
    'n_estimators': [50, 100],
    'max_depth': [8, 14, 20],      # Never leave as None with high sample sizes
    'min_samples_split': [5, 15],
    'max_samples': [0.6, 0.8]      # Subsample training data per tree to save RAM
}

search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=2),
    param_distributions=param_dist,
    n_iter=4,
    scoring='average_precision',
    cv=3,
    n_jobs=2,                      # 2 search workers x 2 estimator threads = 4 threads max
    random_state=42
)
search_rf.fit(X_train, y_train)
best_rf = search_rf.best_estimator_

evaluate_and_record("Random Forest", best_rf, X_test, y_test)

        Model  ROC-AUC   PR-AUC  F1-Score  Precision   Recall
Random Forest 0.930669 0.168796  0.011771    0.00595 0.541772


In [ ]:
from lightgbm import LGBMClassifier

param_dist = {
    'n_estimators': [100, 200],
    'learning_rate': [0.03, 0.1],
    'num_leaves': [31, 63],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# If your OpenCL/LightGBM binary has standard GPU support:
# device='gpu'. If OpenCL is not built, fallback gracefully to CPU with n_jobs=-1.
lgb_model = LGBMClassifier(
    random_state=42,
    device='gpu', 
    verbose=-1
)

search_lgb = RandomizedSearchCV(
    lgb_model,
    param_distributions=param_dist,
    n_iter=6,
    scoring='average_precision',
    cv=3,
    n_jobs=1,   # Keep search at 1 worker so GPU memory isn't contended
    random_state=42
)

try:
    search_lgb.fit(X_train, y_train)
except Exception:
    # Automatic fallback if system OpenCL drivers are missing for LightGBM
    print("LightGBM GPU OpenCL not configured. Falling back to multi-core CPU...")
    search_lgb.estimator.set_params(device='cpu', n_jobs=-1)
    search_lgb.fit(X_train, y_train)

best_lgb = search_lgb.best_estimator_
evaluate_and_record("LightGBM", best_lgb, X_test, y_test)

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/LightGBM/LightGBM/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 860 .

